In [1]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit import transpile
from qiskit.visualization import *
from qiskit.circuit.library import QFT, UnitaryGate
from qiskit.quantum_info import Statevector
from numpy import pi
import numpy as np
from matplotlib import pyplot as plt

# Aer is now a separate package (qiskit-aer)
from qiskit_aer import AerSimulator

In [2]:
Q = 8

In [3]:

N = Q*Q # Total Number of vertex in the grid
l = 4/N # Valule for self loop

In [4]:
# Compute coefficients of the coin
a = 2 / (4 + l)
b = 2 * np.sqrt(l) / (4 + l)
c = 2 * l / (4 + l)

In [5]:
C5 = np.array([
    [a - 1,  a,      a,      a,      b],
    [a,      a - 1,  a,      a,      b],
    [a,      a,      a - 1,  a,      b],
    [a,      a,      a,      a - 1,  b],
    [b,      b,      b,      b,      c - 1]
], dtype=complex)

# Embed into an 8x8 matrix for 3 qubits
C8 = np.zeros((8, 8), dtype=complex)
C8[:5, :5] = C5  # top-left block
for i in range(5, 8):
    C8[i, i] = 1

# Creating the gate for Coin
coin_gate = UnitaryGate(C8, label="Coin")

In [6]:
from qiskit.circuit.library import StatePreparation

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)

In [7]:
coin = QuantumRegister(3,'coin')
nodeX = QuantumRegister(int(np.log2(Q)),'vertex_X')
nodeY = QuantumRegister(int(np.log2(Q)),'vertex_y')
classR = ClassicalRegister(int(2*np.log2(Q)),'measure')

one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')

In [8]:
#phase oracle 
A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code 
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.draw()

┌───┐     ┌───┐
q_0: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_1: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_2: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_3: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_4: ┤ X ├──■──┤ X ├
     ├───┤┌─┴─┐├───┤
q_5: ┤ H ├┤ X ├┤ H ├
     └───┘└───┘└───┘

In [9]:
def superposition(circuit, Q):
    num_states = int(2*np.log2(Q))
    for i in range(0,num_states):
        circuit.h(i)

# Initialization

In [10]:
one_step.append(coin_prep, coin)

In [11]:
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0,seed_transpiler=42)
#one.draw()

In [12]:
one.count_ops()

OrderedDict([('h', 11072), ('t', 6850), ('tdg', 6834), ('cx', 2)])

# COIN 8x8

In [13]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
one_step.append(coin_gate, coin)

In [14]:
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0,seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 297658),
             ('t', 183283),
             ('tdg', 183243),
             ('cx', 19),
             ('x', 2),
             ('s', 2)])

In [15]:
#one.draw(output='mpl', filename='my_circuit.svg')

# Shift

In [16]:
def shift(circuit, Q):
    num_states = 3 + int(2*np.log2(Q))
    circuit.x(num_states-3)
    circuit.x(num_states-2)
    circuit.x(num_states-1)
    E = int(np.log2(Q))
    D = E
    for i in range(int(np.log2(Q))):
        x = list(range(0,E-1))
        x.append(num_states-1)
        x.append(num_states-2)
        x.append(num_states-3)
        circuit.mcx(x,E-1)
        E = E-1
    circuit.x(num_states-3)
    for i in range(int(np.log2(Q))):
        x = list(range(0,E))
        x.append(num_states-1)
        x.append(num_states-2)
        x.append(num_states-3)
        circuit.mcx(x,E)
        E = E+1
    circuit.x(num_states-3)
    circuit.x(num_states-2)
    E = 2*int(np.log2(Q))
    for i in range(int(np.log2(Q))):
        x = list(range(D,E-1))
        x.append(num_states-1)
        x.append(num_states-2)
        x.append(num_states-3)
        circuit.mcx(x,E-1)
        E = E-1
    circuit.x(num_states-3)
    for i in range(int(np.log2(Q))):
        x = list(range(D,E))
        x.append(num_states-1)
        x.append(num_states-2)
        x.append(num_states-3)
        circuit.mcx(x,E)
        E = E+1
    circuit.x(num_states-1)
    circuit.x(num_states-1)
    circuit.mcx([num_states-1],num_states-3)
    circuit.x(num_states-1)

# 8x8

In [17]:
x = 17 #number of steps

In [18]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
superposition(one_step,Q)
one_step.append(coin_prep, coin)
one_step.append(coin_gate,coin)
for i in range(int(x-2)):
    num_states = 3 + int(2*np.log2(Q))
    coin_states = [num_states-3, num_states-2, num_states-1]
    one_step.append(coin_gate,coin_states)
    shift(one_step,Q)
    vertex = list(range(0,num_states-3))
    one_step.append(phase_circuit,vertex)
one_step.measure(vertex,vertex)

In [19]:
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0,seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 4753561),
             ('t', 2933316),
             ('tdg', 2932876),
             ('cx', 6351),
             ('x', 139),
             ('s', 63),
             ('z', 15),
             ('measure', 6)])

In [20]:
one.depth()

6119915

In [ ]:
import json
from qiskit import QuantumCircuit, transpile

# ==========================================
# 1. FAST FAULT-TOLERANT RESOURCE FUNCTION
# ==========================================
def ft_resource_metrics(qc: QuantumCircuit):
    n = qc.num_qubits
    qindex = {q: i for i, q in enumerate(qc.qubits)}

    depth = [0] * n
    tdepth = [0] * n
    cxdepth = [0] * n

    logical_depth = 0
    max_tdepth = 0
    max_cxdepth = 0

    t_count = 0
    cx_count = 0
    measure_count = 0

    for inst in qc.data:
        name = inst.operation.name
        qs = inst.qubits
        nq = len(qs)

        if nq == 0: continue

        if nq == 1:
            q0 = qindex[qs[0]]
            
            # Full depth
            d = depth[q0] + 1
            depth[q0] = d
            if d > logical_depth: logical_depth = d

            # T depth
            td = tdepth[q0]
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            tdepth[q0] = td

            if name == "measure": measure_count += 1

        elif nq == 2:
            q0, q1 = qindex[qs[0]], qindex[qs[1]]
            
            # Full depth
            d0, d1 = depth[q0], depth[q1]
            d = (d0 if d0 > d1 else d1) + 1
            depth[q0] = depth[q1] = d
            if d > logical_depth: logical_depth = d

            # T depth
            td0, td1 = tdepth[q0], tdepth[q1]
            td = td0 if td0 > td1 else td1
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            tdepth[q0] = tdepth[q1] = td

            # CX depth
            cd0, cd1 = cxdepth[q0], cxdepth[q1]
            cd = cd0 if cd0 > cd1 else cd1
            if name == "cx":
                cx_count += 1
                cd += 1
                if cd > max_cxdepth: max_cxdepth = cd
            cxdepth[q0] = cxdepth[q1] = cd

        else:
            inds = [qindex[q] for q in qs]
            
            # Full depth
            d = max(depth[q] for q in inds) + 1
            for q in inds: depth[q] = d
            if d > logical_depth: logical_depth = d

            # T depth
            td = max(tdepth[q] for q in inds)
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            for q in inds: tdepth[q] = td

            # CX depth
            cd = max(cxdepth[q] for q in inds)
            if name == "cx":
                cx_count += 1
                cd += 1
                if cd > max_cxdepth: max_cxdepth = cd
            for q in inds: cxdepth[q] = cd

    return {
        "logical_qubits": n,
        "logical_depth": logical_depth,
        "t_count": t_count,
        "t_depth": max_tdepth,        
        "cx_count": cx_count,
        "cx_depth": max_cxdepth,
        "measurements": measure_count,
    }

# ==========================================
# 2. EXECUTION & PRINTING BLOCK
# ==========================================

one = transpile(
    one_step, 
    basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],
    optimization_level=0,
    seed_transpiler=42
)

print("--- Standard Metrics ---")
print("Total circuit depth (Qiskit):", one.depth())

print("\n--- Fault-Tolerant Metrics ---")
metrics = ft_resource_metrics(one)
print(json.dumps(metrics, indent=4))



--- Standard Metrics ---
Total circuit depth (Qiskit): 6119915

--- Fault-Tolerant Metrics ---
{
    "logical_qubits": 9,
    "logical_depth": 6119915,
    "t_count": 5866192,
    "t_depth": 3382620,
    "cx_count": 6351,
    "cx_depth": 5560,
    "measurements": 6
}


for shift

In [22]:
Q = 8
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
shift(one_step,Q)
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0,seed_transpiler=42)
one.count_ops()

OrderedDict([('cx', 325),
             ('t', 249),
             ('tdg', 236),
             ('h', 136),
             ('x', 8),
             ('s', 2),
             ('z', 1)])

# 16x16

In [23]:
Q = 16
N = Q*Q # Total Number of vertex in the grid
l = 4/N
# Compute coefficients of the coin
a = 2 / (4 + l)
b = 2 * np.sqrt(l) / (4 + l)
c = 2 * l / (4 + l)
C5 = np.array([
    [a - 1,  a,      a,      a,      b],
    [a,      a - 1,  a,      a,      b],
    [a,      a,      a - 1,  a,      b],
    [a,      a,      a,      a - 1,  b],
    [b,      b,      b,      b,      c - 1]
], dtype=complex)

# Embed into an 8x8 matrix for 3 qubits
C8 = np.zeros((8, 8), dtype=complex)
C8[:5, :5] = C5  # top-left block
for i in range(5, 8):
    C8[i, i] = 1

# Creating the gate for Coin
coin_gate = UnitaryGate(C8, label="Coin")
from qiskit.circuit.library import StatePreparation

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)
coin = QuantumRegister(3,'coin')
nodeX = QuantumRegister(int(np.log2(Q)),'vertex_X')
nodeY = QuantumRegister(int(np.log2(Q)),'vertex_y')
classR = ClassicalRegister(int(2*np.log2(Q)),'measure')

one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')


In [24]:
#phase oracle 
A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code 
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.draw()

┌───┐     ┌───┐
q_0: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_1: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_2: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_3: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_4: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_5: ┤ X ├──■──┤ X ├
     ├───┤  │  ├───┤
q_6: ┤ X ├──■──┤ X ├
     ├───┤┌─┴─┐├───┤
q_7: ┤ H ├┤ X ├┤ H ├
     └───┘└───┘└───┘

In [25]:
x = 33
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
superposition(one_step,Q)
one_step.append(coin_prep, coin)
one_step.append(coin_gate,coin)
for i in range(int(x-2)):
    num_states = 3 + int(2*np.log2(Q))
    coin_states = [num_states-3, num_states-2, num_states-1]
    one_step.append(coin_gate,coin_states)
    shift(one_step,Q)
    vertex = list(range(0,num_states-3))
    one_step.append(phase_circuit,vertex)
one_step.measure(vertex,vertex)

In [26]:
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0,seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 9071189),
             ('t', 5598425),
             ('tdg', 5597907),
             ('cx', 20543),
             ('x', 378),
             ('s', 158),
             ('z', 62),
             ('measure', 8),
             ('sdg', 1)])

In [27]:
one.depth()

12094507

In [ ]:
import json
from qiskit import QuantumCircuit, transpile

# ==========================================
# 1. FAST FAULT-TOLERANT RESOURCE FUNCTION
# ==========================================
def ft_resource_metrics(qc: QuantumCircuit):
    n = qc.num_qubits
    qindex = {q: i for i, q in enumerate(qc.qubits)}

    depth = [0] * n
    tdepth = [0] * n
    cxdepth = [0] * n

    logical_depth = 0
    max_tdepth = 0
    max_cxdepth = 0

    t_count = 0
    cx_count = 0
    measure_count = 0

    for inst in qc.data:
        name = inst.operation.name
        qs = inst.qubits
        nq = len(qs)

        if nq == 0: continue

        if nq == 1:
            q0 = qindex[qs[0]]
            
            # Full depth
            d = depth[q0] + 1
            depth[q0] = d
            if d > logical_depth: logical_depth = d

            # T depth
            td = tdepth[q0]
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            tdepth[q0] = td

            if name == "measure": measure_count += 1

        elif nq == 2:
            q0, q1 = qindex[qs[0]], qindex[qs[1]]
            
            # Full depth
            d0, d1 = depth[q0], depth[q1]
            d = (d0 if d0 > d1 else d1) + 1
            depth[q0] = depth[q1] = d
            if d > logical_depth: logical_depth = d

            # T depth
            td0, td1 = tdepth[q0], tdepth[q1]
            td = td0 if td0 > td1 else td1
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            tdepth[q0] = tdepth[q1] = td

            # CX depth
            cd0, cd1 = cxdepth[q0], cxdepth[q1]
            cd = cd0 if cd0 > cd1 else cd1
            if name == "cx":
                cx_count += 1
                cd += 1
                if cd > max_cxdepth: max_cxdepth = cd
            cxdepth[q0] = cxdepth[q1] = cd

        else:
            inds = [qindex[q] for q in qs]
            
            # Full depth
            d = max(depth[q] for q in inds) + 1
            for q in inds: depth[q] = d
            if d > logical_depth: logical_depth = d

            # T depth
            td = max(tdepth[q] for q in inds)
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            for q in inds: tdepth[q] = td

            # CX depth
            cd = max(cxdepth[q] for q in inds)
            if name == "cx":
                cx_count += 1
                cd += 1
                if cd > max_cxdepth: max_cxdepth = cd
            for q in inds: cxdepth[q] = cd

    return {
        "logical_qubits": n,
        "logical_depth": logical_depth,
        "t_count": t_count,
        "t_depth": max_tdepth,        
        "cx_count": cx_count,
        "cx_depth": max_cxdepth,
        "measurements": measure_count,
    }

# ==========================================
# 2. EXECUTION & PRINTING BLOCK
# ==========================================

one = transpile(
    one_step, 
    basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],
    optimization_level=0 ,
    seed_transpiler=42
)

print("--- Standard Metrics ---")
print("Total circuit depth (Qiskit):", one.depth())

print("\n--- Fault-Tolerant Metrics ---")
metrics = ft_resource_metrics(one)
print(json.dumps(metrics, indent=4))


--- Standard Metrics ---
Total circuit depth (Qiskit): 12094507

--- Fault-Tolerant Metrics ---
{
    "logical_qubits": 11,
    "logical_depth": 12094507,
    "t_count": 11196332,
    "t_depth": 6677094,
    "cx_count": 20543,
    "cx_depth": 17541,
    "measurements": 8
}


for shift

In [29]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
shift(one_step,Q)
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0,seed_transpiler=42)
one.count_ops()

OrderedDict([('cx', 511),
             ('t', 390),
             ('tdg', 375),
             ('h', 224),
             ('x', 8),
             ('s', 2),
             ('z', 2)])

# 32x32

In [30]:
Q = 32
N = Q*Q # Total Number of vertex in the grid
l = 4/N
# Compute coefficients of the coin
a = 2 / (4 + l)
b = 2 * np.sqrt(l) / (4 + l)
c = 2 * l / (4 + l)
C5 = np.array([
    [a - 1,  a,      a,      a,      b],
    [a,      a - 1,  a,      a,      b],
    [a,      a,      a - 1,  a,      b],
    [a,      a,      a,      a - 1,  b],
    [b,      b,      b,      b,      c - 1]
], dtype=complex)

# Embed into an 8x8 matrix for 3 qubits
C8 = np.zeros((8, 8), dtype=complex)
C8[:5, :5] = C5  # top-left block
for i in range(5, 8):
    C8[i, i] = 1

# Creating the gate for Coin
coin_gate = UnitaryGate(C8, label="Coin")
from qiskit.circuit.library import StatePreparation

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)
coin = QuantumRegister(3,'coin')
nodeX = QuantumRegister(int(np.log2(Q)),'vertex_X')
nodeY = QuantumRegister(int(np.log2(Q)),'vertex_y')
classR = ClassicalRegister(int(2*np.log2(Q)),'measure')

one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')


In [31]:
#phase oracle 
A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code 
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
#phase_circuit.draw()

In [32]:
x = 75
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
superposition(one_step,Q)
one_step.append(coin_prep, coin)
one_step.append(coin_gate,coin)
for i in range(int(x-2)):
    num_states = 3 + int(2*np.log2(Q))
    coin_states = [num_states-3, num_states-2, num_states-1]
    one_step.append(coin_gate,coin_states)
    shift(one_step,Q)
    vertex = list(range(0,num_states-3))
    one_step.append(phase_circuit,vertex)
one_step.measure(vertex,vertex)

In [33]:
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0,seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 21306752),
             ('t', 13132517),
             ('tdg', 13130368),
             ('cx', 69371),
             ('x', 1174),
             ('s', 441),
             ('z', 147),
             ('measure', 10),
             ('sdg', 1)])

In [34]:
one.depth()

28574954

In [35]:
import json
from qiskit import QuantumCircuit, transpile

# ==========================================
# 1. FAST FAULT-TOLERANT RESOURCE FUNCTION
# ==========================================
def ft_resource_metrics(qc: QuantumCircuit):
    n = qc.num_qubits
    qindex = {q: i for i, q in enumerate(qc.qubits)}

    depth = [0] * n
    tdepth = [0] * n
    cxdepth = [0] * n

    logical_depth = 0
    max_tdepth = 0
    max_cxdepth = 0

    t_count = 0
    cx_count = 0
    measure_count = 0

    for inst in qc.data:
        name = inst.operation.name
        qs = inst.qubits
        nq = len(qs)

        if nq == 0: continue

        if nq == 1:
            q0 = qindex[qs[0]]
            
            # Full depth
            d = depth[q0] + 1
            depth[q0] = d
            if d > logical_depth: logical_depth = d

            # T depth
            td = tdepth[q0]
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            tdepth[q0] = td

            if name == "measure": measure_count += 1

        elif nq == 2:
            q0, q1 = qindex[qs[0]], qindex[qs[1]]
            
            # Full depth
            d0, d1 = depth[q0], depth[q1]
            d = (d0 if d0 > d1 else d1) + 1
            depth[q0] = depth[q1] = d
            if d > logical_depth: logical_depth = d

            # T depth
            td0, td1 = tdepth[q0], tdepth[q1]
            td = td0 if td0 > td1 else td1
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            tdepth[q0] = tdepth[q1] = td

            # CX depth
            cd0, cd1 = cxdepth[q0], cxdepth[q1]
            cd = cd0 if cd0 > cd1 else cd1
            if name == "cx":
                cx_count += 1
                cd += 1
                if cd > max_cxdepth: max_cxdepth = cd
            cxdepth[q0] = cxdepth[q1] = cd

        else:
            inds = [qindex[q] for q in qs]
            
            # Full depth
            d = max(depth[q] for q in inds) + 1
            for q in inds: depth[q] = d
            if d > logical_depth: logical_depth = d

            # T depth
            td = max(tdepth[q] for q in inds)
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            for q in inds: tdepth[q] = td

            # CX depth
            cd = max(cxdepth[q] for q in inds)
            if name == "cx":
                cx_count += 1
                cd += 1
                if cd > max_cxdepth: max_cxdepth = cd
            for q in inds: cxdepth[q] = cd

    return {
        "logical_qubits": n,
        "logical_depth": logical_depth,
        "t_count": t_count,
        "t_depth": max_tdepth,        
        "cx_count": cx_count,
        "cx_depth": max_cxdepth,
        "measurements": measure_count,
    }

# ==========================================
# 2. EXECUTION & PRINTING BLOCK
# ==========================================

one = transpile(
    one_step, 
    basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],
    optimization_level=0 ,
    seed_transpiler=42
)

print("--- Standard Metrics ---")
print("Total circuit depth (Qiskit):", one.depth())

print("\n--- Fault-Tolerant Metrics ---")
metrics = ft_resource_metrics(one)
print(json.dumps(metrics, indent=4))


--- Standard Metrics ---
Total circuit depth (Qiskit): 28574954

--- Fault-Tolerant Metrics ---
{
    "logical_qubits": 13,
    "logical_depth": 28574954,
    "t_count": 26262885,
    "t_depth": 15745434,
    "cx_count": 69371,
    "cx_depth": 55434,
    "measurements": 10
}


for shift


In [36]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
shift(one_step,Q)
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0,seed_transpiler=42)
one.count_ops()

OrderedDict([('cx', 745),
             ('t', 579),
             ('tdg', 559),
             ('h', 342),
             ('x', 8),
             ('s', 3),
             ('z', 2)])

# 64x64

In [37]:
Q = 64
N = Q*Q # Total Number of vertex in the grid
l = 4/N
# Compute coefficients of the coin
a = 2 / (4 + l)
b = 2 * np.sqrt(l) / (4 + l)
c = 2 * l / (4 + l)
C5 = np.array([
    [a - 1,  a,      a,      a,      b],
    [a,      a - 1,  a,      a,      b],
    [a,      a,      a - 1,  a,      b],
    [a,      a,      a,      a - 1,  b],
    [b,      b,      b,      b,      c - 1]
], dtype=complex)

# Embed into an 8x8 matrix for 3 qubits
C8 = np.zeros((8, 8), dtype=complex)
C8[:5, :5] = C5  # top-left block
for i in range(5, 8):
    C8[i, i] = 1

# Creating the gate for Coin
coin_gate = UnitaryGate(C8, label="Coin")
from qiskit.circuit.library import StatePreparation

coin_init = np.zeros(8, dtype=complex)
coin_init[0] = 1 / np.sqrt(4 + l)           # 000
coin_init[1] = 1 / np.sqrt(4 + l)           # 001
coin_init[2] = 1 / np.sqrt(4 + l)           # 010
coin_init[3] = 1 / np.sqrt(4 + l)           # 011
coin_init[4] = np.sqrt(l) / np.sqrt(4 + l)  # 100

coin_prep = StatePreparation(coin_init)
coin = QuantumRegister(3,'coin')
nodeX = QuantumRegister(int(np.log2(Q)),'vertex_X')
nodeY = QuantumRegister(int(np.log2(Q)),'vertex_y')
classR = ClassicalRegister(int(2*np.log2(Q)),'measure')

one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')


In [38]:
#phase oracle 
A = 2 * int(np.log2(Q))
phase_circuit =  QuantumCircuit(A, name=' phase oracle ')
# Mark 100000 for any code 
cont = []
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
phase_circuit.h(A-1)
phase_circuit.mcx(cont,A-1)
phase_circuit.h(A-1)
for i in range(0,A-1):
    phase_circuit.x(i)
    cont.append(i)
#phase_circuit.draw()

In [39]:
x = 165
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
superposition(one_step,Q)
one_step.append(coin_prep, coin)
one_step.append(coin_gate,coin)
for i in range(int(x-2)):
    num_states = 3 + int(2*np.log2(Q))
    coin_states = [num_states-3, num_states-2, num_states-1]
    one_step.append(coin_gate,coin_states)
    shift(one_step,Q)
    vertex = list(range(0,num_states-3))
    one_step.append(phase_circuit,vertex)
one_step.measure(vertex,vertex)

In [40]:
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0,seed_transpiler=42)
one.count_ops()

OrderedDict([('h', 47408197),
             ('t', 29250155),
             ('tdg', 29244911),
             ('cx', 209639),
             ('x', 3265),
             ('s', 1144),
             ('z', 326),
             ('measure', 12),
             ('sdg', 1)])

In [41]:
one.depth()

63891729

In [42]:
import json
from qiskit import QuantumCircuit, transpile

# ==========================================
# 1. FAST FAULT-TOLERANT RESOURCE FUNCTION
# ==========================================
def ft_resource_metrics(qc: QuantumCircuit):
    n = qc.num_qubits
    qindex = {q: i for i, q in enumerate(qc.qubits)}

    depth = [0] * n
    tdepth = [0] * n
    cxdepth = [0] * n

    logical_depth = 0
    max_tdepth = 0
    max_cxdepth = 0

    t_count = 0
    cx_count = 0
    measure_count = 0

    for inst in qc.data:
        name = inst.operation.name
        qs = inst.qubits
        nq = len(qs)

        if nq == 0: continue

        if nq == 1:
            q0 = qindex[qs[0]]
            
            # Full depth
            d = depth[q0] + 1
            depth[q0] = d
            if d > logical_depth: logical_depth = d

            # T depth
            td = tdepth[q0]
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            tdepth[q0] = td

            if name == "measure": measure_count += 1

        elif nq == 2:
            q0, q1 = qindex[qs[0]], qindex[qs[1]]
            
            # Full depth
            d0, d1 = depth[q0], depth[q1]
            d = (d0 if d0 > d1 else d1) + 1
            depth[q0] = depth[q1] = d
            if d > logical_depth: logical_depth = d

            # T depth
            td0, td1 = tdepth[q0], tdepth[q1]
            td = td0 if td0 > td1 else td1
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            tdepth[q0] = tdepth[q1] = td

            # CX depth
            cd0, cd1 = cxdepth[q0], cxdepth[q1]
            cd = cd0 if cd0 > cd1 else cd1
            if name == "cx":
                cx_count += 1
                cd += 1
                if cd > max_cxdepth: max_cxdepth = cd
            cxdepth[q0] = cxdepth[q1] = cd

        else:
            inds = [qindex[q] for q in qs]
            
            # Full depth
            d = max(depth[q] for q in inds) + 1
            for q in inds: depth[q] = d
            if d > logical_depth: logical_depth = d

            # T depth
            td = max(tdepth[q] for q in inds)
            if name in {"t", "tdg"}:
                t_count += 1
                td += 1
                if td > max_tdepth: max_tdepth = td
            for q in inds: tdepth[q] = td

            # CX depth
            cd = max(cxdepth[q] for q in inds)
            if name == "cx":
                cx_count += 1
                cd += 1
                if cd > max_cxdepth: max_cxdepth = cd
            for q in inds: cxdepth[q] = cd

    return {
        "logical_qubits": n,
        "logical_depth": logical_depth,
        "t_count": t_count,
        "t_depth": max_tdepth,        
        "cx_count": cx_count,
        "cx_depth": max_cxdepth,
        "measurements": measure_count,
    }

# ==========================================
# 2. EXECUTION & PRINTING BLOCK
# ==========================================

one = transpile(
    one_step, 
    basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],
    optimization_level=0 ,
    seed_transpiler=42
)

print("--- Standard Metrics ---")
print("Total circuit depth (Qiskit):", one.depth())

print("\n--- Fault-Tolerant Metrics ---")
metrics = ft_resource_metrics(one)
print(json.dumps(metrics, indent=4))


--- Standard Metrics ---
Total circuit depth (Qiskit): 63891729

--- Fault-Tolerant Metrics ---
{
    "logical_qubits": 15,
    "logical_depth": 63891729,
    "t_count": 58495066,
    "t_depth": 35191260,
    "cx_count": 209639,
    "cx_depth": 158626,
    "measurements": 12
}


In [43]:
one_step = QuantumCircuit(nodeX, nodeY, coin,  classR, name='ONE_STEP')
shift(one_step,Q)
one = transpile(one_step, basis_gates=["x","z","h","s","sdg", "cx","t", "tdg"],optimization_level=0,seed_transpiler=42)
one.count_ops()

OrderedDict([('cx', 1027),
             ('t', 813),
             ('tdg', 789),
             ('h', 490),
             ('x', 8),
             ('s', 5),
             ('z', 2)])